# Jalon 2 — Analyse Exploratoire & Évaluation de la Baseline

**Projet** : DocuFlow AI — Assistant intelligent pour le traitement documentaire  
**Dataset** : SROIE (ICDAR 2019 — Scanned Receipts OCR and Information Extraction)  
**Objectif** : Évaluer notre pipeline actuel (Tesseract OCR + extraction regex) sur un benchmark reconnu.

---

## Table des matières
1. Chargement du dataset
2. Analyse exploratoire (EDA)
3. Évaluation de la baseline (OCR + Extraction)
4. Calcul des métriques (EM, F1, CER, WER)
5. Export des résultats

In [ ]:
import sys
import json
from pathlib import Path
from collections import Counter

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# Ajout du projet au path pour importer nos modules
PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from docuflow_ai.modules.ocr_engine import ocr_image
from docuflow_ai.modules.entity_extractor import extract_entities, compute_extraction_score
from docuflow_ai.modules.metrics import compute_cer, compute_wer

# SROIE est en anglais : on force lang="eng" pour éviter l'erreur si "fra" n'est pas installé
OCR_LANG = "eng"

def process_document_baseline(file_path: str) -> dict:
    """Wrapper OCR pour la baseline (langue anglaise pour SROIE)."""
    return ocr_image(file_path, lang=OCR_LANG)

DATASET_DIR = Path("dataset")
print(f"Racine projet : {PROJECT_ROOT}")
print(f"Dataset dir   : {DATASET_DIR.resolve()}")

---
## 1. Chargement du dataset

On charge le manifest du split **test** pour l'évaluation, et un aperçu du split train pour l'EDA.

In [ ]:
def load_split(split_name: str) -> list:
    """Charge le manifest d'un split."""
    manifest_path = DATASET_DIR / split_name / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(
            f"Manifest introuvable : {manifest_path}\n"
            "Exécutez d'abord : python prepare_dataset.py"
        )
    return json.loads(manifest_path.read_text(encoding="utf-8"))

train_data = load_split("train")
val_data = load_split("validation")
test_data = load_split("test")

print(f"Train      : {len(train_data)} documents")
print(f"Validation : {len(val_data)} documents")
print(f"Test       : {len(test_data)} documents")
print(f"Total      : {len(train_data) + len(val_data) + len(test_data)} documents")

---
## 2. Analyse Exploratoire (EDA)

### 2.1 Visualisation d'exemples de documents

In [ ]:
# Affichage de 4 exemples de reçus avec leur vérité terrain
fig, axes = plt.subplots(1, 4, figsize=(16, 6))
fig.suptitle("Exemples de reçus SROIE (split train)", fontsize=14)

for idx, ax in enumerate(axes):
    if idx >= len(train_data):
        ax.axis("off")
        continue
    doc = train_data[idx]
    img_path = DATASET_DIR / "train" / doc["image"]
    img = Image.open(img_path)
    ax.imshow(img, cmap="gray")
    gt = doc["ground_truth"]
    title = f"ID: {doc['id']}\n"
    if "company" in gt:
        title += f"{gt['company'][:20]}\n"
    if "total" in gt:
        title += f"Total: {gt['total']}"
    ax.set_title(title, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig("eda_samples.png", dpi=100, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : eda_samples.png")

### 2.2 Statistiques sur les annotations (vérité terrain)

In [ ]:
# Analyse des champs disponibles dans la vérité terrain
all_data = train_data + val_data + test_data

field_presence = Counter()
field_lengths = {"company": [], "date": [], "address": [], "total": []}

for doc in all_data:
    gt = doc["ground_truth"]
    for field in field_lengths:
        if field in gt and gt[field]:
            field_presence[field] += 1
            field_lengths[field].append(len(gt[field]))

print("=== Présence des champs dans la vérité terrain ===")
print(f"{'Champ':<12} {'Présent':>8} {'%':>8} {'Long. moy.':>12}")
print("-" * 44)
for field in field_lengths:
    count = field_presence[field]
    pct = 100 * count / len(all_data) if all_data else 0
    avg_len = sum(field_lengths[field]) / len(field_lengths[field]) if field_lengths[field] else 0
    print(f"{field:<12} {count:>8} {pct:>7.1f}% {avg_len:>11.1f}")

In [ ]:
# Distribution de la longueur du texte OCR de référence
ocr_lengths = [len(doc.get("ocr_reference", "")) for doc in all_data]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ocr_lengths, bins=20, edgecolor="black", alpha=0.7, color="steelblue")
axes[0].set_xlabel("Nombre de caractères (texte OCR référence)")
axes[0].set_ylabel("Nombre de documents")
axes[0].set_title("Distribution de la longueur du texte OCR")
axes[0].axvline(x=sum(ocr_lengths)/len(ocr_lengths), color="red", linestyle="--", label="Moyenne")
axes[0].legend()

# Taille des images
widths, heights = [], []
for doc in train_data[:20]:
    img_path = DATASET_DIR / "train" / doc["image"]
    if img_path.exists():
        img = Image.open(img_path)
        widths.append(img.width)
        heights.append(img.height)

axes[1].scatter(widths, heights, alpha=0.6, color="darkorange")
axes[1].set_xlabel("Largeur (px)")
axes[1].set_ylabel("Hauteur (px)")
axes[1].set_title("Dimensions des images")

plt.tight_layout()
plt.savefig("eda_distributions.png", dpi=100, bbox_inches="tight")
plt.show()

### 2.3 Défis identifiés dans les données

| Défi | Description | Impact sur notre pipeline |
|------|-------------|---------------------------|
| **Qualité variable des scans** | Certains reçus sont flous, mal cadrés ou à faible contraste | Augmente le CER/WER de Tesseract |
| **Mises en page non-standard** | Colonnes, alignements variés, tableaux implicites | L'OCR peut mélanger l'ordre de lecture |
| **Polices thermiques** | Texte imprimé sur papier thermique (reçus de caisse) | Caractères parfois dégradés |
| **Formats de date/montant variés** | Formats anglophones vs francophones (MM/DD vs DD/MM, . vs ,) | Nos regex sont calibrées pour le français |
| **Bruit et artefacts** | Pliures, taches, arrière-plans texturés | Faux positifs OCR possibles |
| **Multilinguisme** | Reçus en anglais/chinois dans SROIE | Notre pipeline est optimisé FR+EN |

---
## 3. Évaluation de la Baseline

Notre baseline est le pipeline actuel de DocuFlow AI :
1. **OCR** : Tesseract 5.4 (fra+eng)
2. **Extraction d'entités** : Regex Python (dates, montants, emails, etc.)

On évalue sur le split **test**.

In [ ]:
import time

results = []
ocr_errors_cer = []
ocr_errors_wer = []
extraction_scores = []
processing_times = []

print(f"Évaluation de la baseline sur {len(test_data)} documents (split test)...")
print("-" * 60)

for i, doc in enumerate(test_data):
    img_path = DATASET_DIR / "test" / doc["image"]
    if not img_path.exists():
        print(f"  [skip] {doc['id']} — image introuvable")
        continue

    # 1. OCR avec notre pipeline (baseline)
    t0 = time.time()
    ocr_result = process_document_baseline(str(img_path.resolve()))
    t_ocr = time.time() - t0
    processing_times.append(t_ocr)

    predicted_text = ocr_result.get("text", "")
    confidence = ocr_result.get("confidence", 0)

    # 2. CER / WER vs vérité terrain OCR
    reference_text = doc.get("ocr_reference", "")
    if reference_text:
        cer = compute_cer(predicted_text, reference_text)
        wer = compute_wer(predicted_text, reference_text)
        ocr_errors_cer.append(cer)
        ocr_errors_wer.append(wer)
    else:
        cer, wer = None, None

    # 3. Extraction d'entités
    extracted = extract_entities(predicted_text)

    # 4. Comparaison avec la vérité terrain (mapping SROIE -> notre format)
    gt = doc["ground_truth"]
    gt_mapped = {}
    if "date" in gt:
        gt_mapped["dates"] = [gt["date"]]
    if "total" in gt:
        gt_mapped["montants"] = [gt["total"]]

    if gt_mapped:
        score = compute_extraction_score(extracted, gt_mapped)
        extraction_scores.append(score)

    results.append({
        "id": doc["id"],
        "ocr_confidence": confidence,
        "ocr_time_s": round(t_ocr, 3),
        "cer": cer,
        "wer": wer,
        "entities_found": sum(len(v) for v in extracted.values()),
        "gt_fields": list(gt.keys()),
    })

    if (i + 1) % 5 == 0 or i == len(test_data) - 1:
        cer_str = f"{cer:.3f}" if cer is not None else "N/A"
        wer_str = f"{wer:.3f}" if wer is not None else "N/A"
        print(f"  [{i+1}/{len(test_data)}] {doc['id']} — CER={cer_str}, "
              f"WER={wer_str}, conf={confidence:.1f}%, t={t_ocr:.2f}s")

print("\nEvaluation terminee.")

---
## 4. Calcul des Métriques

Métriques exigées pour le Jalon 2 :
- **CER** (Character Error Rate) — cible < 5%
- **WER** (Word Error Rate) — cible < 10%
- **Exact Match (EM)** — cible > 80%
- **F1-Score entités** — cible > 85%

In [ ]:
import numpy as np

# --- Métriques OCR ---
avg_cer = np.mean(ocr_errors_cer) if ocr_errors_cer else float("nan")
avg_wer = np.mean(ocr_errors_wer) if ocr_errors_wer else float("nan")
median_cer = np.median(ocr_errors_cer) if ocr_errors_cer else float("nan")
median_wer = np.median(ocr_errors_wer) if ocr_errors_wer else float("nan")

# --- Métriques Extraction ---
em_scores = [s["exact_match"] for s in extraction_scores] if extraction_scores else []
f1_scores = [s["f1_score"] for s in extraction_scores] if extraction_scores else []
precision_scores = [s["precision"] for s in extraction_scores] if extraction_scores else []
recall_scores = [s["recall"] for s in extraction_scores] if extraction_scores else []

avg_em = np.mean(em_scores) if em_scores else float("nan")
avg_f1 = np.mean(f1_scores) if f1_scores else float("nan")
avg_precision = np.mean(precision_scores) if precision_scores else float("nan")
avg_recall = np.mean(recall_scores) if recall_scores else float("nan")

# --- Temps de traitement ---
avg_time = np.mean(processing_times) if processing_times else float("nan")

print("=" * 60)
print("  MÉTRIQUES DE LA BASELINE — DocuFlow AI (Jalon 2)")
print("=" * 60)
print(f"\n{'Métrique':<25} {'Valeur':>10} {'Cible':>10} {'Statut':>10}")
print("-" * 58)

def status(value, target, lower_is_better=True):
    if np.isnan(value):
        return "N/A"
    if lower_is_better:
        return "✓" if value <= target else "✗"
    return "✓" if value >= target else "✗"

metrics_display = [
    ("CER (moy.)", avg_cer, 0.05, True),
    ("WER (moy.)", avg_wer, 0.10, True),
    ("Exact Match", avg_em, 0.80, False),
    ("F1-Score entités", avg_f1, 0.85, False),
    ("Précision", avg_precision, 0.80, False),
    ("Rappel", avg_recall, 0.80, False),
    ("Temps moy./doc (s)", avg_time, 10.0, True),
]

for name, value, target, lower in metrics_display:
    val_str = f"{value:.4f}" if not np.isnan(value) else "N/A"
    tgt_str = f"{'<' if lower else '>'} {target}"
    st = status(value, target, lower)
    print(f"{name:<25} {val_str:>10} {tgt_str:>10} {st:>10}")

print(f"\nDocuments évalués : {len(results)}")
print(f"Documents avec réf. OCR : {len(ocr_errors_cer)}")
print(f"Documents avec GT entités : {len(extraction_scores)}")

In [ ]:
# Visualisation des métriques
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Métriques Baseline — DocuFlow AI", fontsize=14)

# CER distribution
if ocr_errors_cer:
    axes[0, 0].hist(ocr_errors_cer, bins=15, edgecolor="black", alpha=0.7, color="#e74c3c")
    axes[0, 0].axvline(x=0.05, color="green", linestyle="--", label="Cible (5%)")
    axes[0, 0].axvline(x=avg_cer, color="blue", linestyle="-", label=f"Moyenne ({avg_cer:.3f})")
    axes[0, 0].set_xlabel("CER")
    axes[0, 0].set_title("Distribution du CER")
    axes[0, 0].legend()

# WER distribution
if ocr_errors_wer:
    axes[0, 1].hist(ocr_errors_wer, bins=15, edgecolor="black", alpha=0.7, color="#3498db")
    axes[0, 1].axvline(x=0.10, color="green", linestyle="--", label="Cible (10%)")
    axes[0, 1].axvline(x=avg_wer, color="blue", linestyle="-", label=f"Moyenne ({avg_wer:.3f})")
    axes[0, 1].set_xlabel("WER")
    axes[0, 1].set_title("Distribution du WER")
    axes[0, 1].legend()

# F1 par document
if f1_scores:
    axes[1, 0].bar(range(len(f1_scores)), f1_scores, color="#2ecc71", alpha=0.7)
    axes[1, 0].axhline(y=0.85, color="red", linestyle="--", label="Cible (85%)")
    axes[1, 0].axhline(y=avg_f1, color="blue", linestyle="-", label=f"Moyenne ({avg_f1:.3f})")
    axes[1, 0].set_xlabel("Document")
    axes[1, 0].set_ylabel("F1-Score")
    axes[1, 0].set_title("F1-Score entités par document")
    axes[1, 0].legend()

# Temps de traitement
if processing_times:
    axes[1, 1].bar(range(len(processing_times)), processing_times, color="#9b59b6", alpha=0.7)
    axes[1, 1].axhline(y=10.0, color="red", linestyle="--", label="Cible (10s)")
    axes[1, 1].axhline(y=avg_time, color="blue", linestyle="-", label=f"Moyenne ({avg_time:.2f}s)")
    axes[1, 1].set_xlabel("Document")
    axes[1, 1].set_ylabel("Temps (s)")
    axes[1, 1].set_title("Temps de traitement par document")
    axes[1, 1].legend()

plt.tight_layout()
plt.savefig("baseline_metrics_charts.png", dpi=100, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : baseline_metrics_charts.png")

---
## 5. Export des résultats

Sauvegarde du résumé des métriques dans `baseline_metrics.json`.

In [ ]:
def safe_float(v):
    """Convertit les NaN en None pour la sérialisation JSON."""
    if isinstance(v, float) and np.isnan(v):
        return None
    return round(float(v), 4) if isinstance(v, (float, np.floating)) else v

baseline_metrics = {
    "project": "DocuFlow AI",
    "jalon": "Jalon 2 — Baseline Evaluation",
    "dataset": "SROIE (ICDAR 2019)",
    "split": "test",
    "n_documents": len(results),
    "pipeline": {
        "ocr": "Tesseract 5.4 (fra+eng)",
        "extraction": "Regex Python (entity_extractor.py)",
        "llm": "N/A (baseline sans LLM)"
    },
    "metrics": {
        "ocr": {
            "cer_mean": safe_float(avg_cer),
            "cer_median": safe_float(median_cer),
            "wer_mean": safe_float(avg_wer),
            "wer_median": safe_float(median_wer),
            "n_evaluated": len(ocr_errors_cer)
        },
        "extraction": {
            "exact_match": safe_float(avg_em),
            "f1_score": safe_float(avg_f1),
            "precision": safe_float(avg_precision),
            "recall": safe_float(avg_recall),
            "n_evaluated": len(extraction_scores)
        },
        "performance": {
            "avg_time_per_doc_s": safe_float(avg_time),
            "total_time_s": safe_float(sum(processing_times)) if processing_times else None
        }
    },
    "targets": {
        "cer": "< 0.05",
        "wer": "< 0.10",
        "exact_match": "> 0.80",
        "f1_score": "> 0.85",
        "time_per_doc": "< 10s"
    },
    "per_document": results
}

output_path = Path("baseline_metrics.json")
output_path.write_text(
    json.dumps(baseline_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"✓ Métriques exportées dans : {output_path.resolve()}")
print(f"\nRésumé :")
print(f"  CER moyen  : {safe_float(avg_cer)}")
print(f"  WER moyen  : {safe_float(avg_wer)}")
print(f"  Exact Match: {safe_float(avg_em)}")
print(f"  F1-Score   : {safe_float(avg_f1)}")
print(f"  Temps moy. : {safe_float(avg_time)}s / document")

---
## Conclusion & Prochaines étapes

### Analyse de la baseline

La baseline actuelle (Tesseract + Regex) montre les limitations attendues :
- Le **CER/WER** dépend fortement de la qualité du scan et de la mise en page
- L'**extraction regex** est conçue pour des formats français ; le dataset SROIE étant anglophone, les performances d'extraction seront naturellement plus basses
- Le **temps de traitement** reste dans les cibles (< 10s/doc)

### Améliorations prévues (Jalon 3)
1. **Pré-traitement d'image** : binarisation, correction d'inclinaison
2. **Extraction par LLM** : utiliser Gemma 4 via Ollama pour extraire les champs structurés
3. **Prompts adaptatifs** : adapter le prompt LLM selon le type de document détecté
4. **Augmentation** : enrichir le jeu de test avec des documents français réels